<a href="https://colab.research.google.com/github/cankat-byte/DATA_SCIENCE_project/blob/main/MARPOL_RAG_CHATBOT_3_hatal%C4%B1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1.Hücre - Hazırlık

In [1]:
# Gerekli tüm güncel ve uyumlu paketler (PyMuPDF desteği ile)
!pip install -q -U langchain langchain-community langchain-groq langchain-huggingface langchain-chroma chromadb pypdf pymupdf streamlit

2.Hücre - Kernel yeniden başlatma ( Çalışma zamanı yeniden başlatmak için. DİKKAT sadece hazırlık aşaması sonrası çalıştır)

In [2]:
# ⚠️ DİKKAT: Bu hücreyi SADECE paket kurulumu bittiğinde 1 KERE çalıştır.

import IPython
IPython.display.display(IPython.display.HTML("<script>google.colab.kernel.restart()</script>"))


3.Hücre - Groq API anahtarı tanımlama

In [4]:
import os
from google.colab import userdata

# Colab sol panelindeki 🔑 simgesine tıklayıp "GROQ_API_KEY" adıyla anahtarı ekle
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')


**4.Hücre - Kütüphanelerin Yüklenmesi** ( bu adımı gereksiz olacağı için atladım , streamlit kendi hücresinde zaten kütüphaneleri okuyacak / colab içinde arayüz oluşturacağımda başka bir hücrede tekrar gerekli kütüphaneleri yazacğım)

5.Hücre - GROQ Kurulum (uygun model seçimi)

In [5]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.1)

6.Hücre - PDF parçalama / Veritabanı kayıt

In [9]:
import os
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# ==========================================
# 1. PDF Belgesini Yükleme
# ==========================================
pdf_yolu = "/content/sozlesme.pdf"

if not os.path.exists(pdf_yolu):
    raise FileNotFoundError(f"HATA: '{pdf_yolu}' konumunda PDF dosyası bulunamadı! Lütfen Colab'e yükleyin.")

loader = PyMuPDFLoader(pdf_yolu)
documents = loader.load()

# ==========================================
# 2. Metni Parçalara Bölme
# ==========================================
text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=200)
splits = text_splitter.split_documents(documents)

# ==========================================
# 3. Vektör Veritabanı ve Retriever Oluşturma
# ==========================================
# Güncel kütüphane üzerinden gömme (embedding) modeli çağrılır
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/distiluse-base-multilingual-cased-v1")

# Bellek içi (InMemory) vektör deposu splits oluştuktan sonra beslenir
vectorstore = InMemoryVectorStore.from_documents(splits, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})  # En alakalı 3 parçayı getirir

# ==========================================
# 4. Prompt (Şablon) Tanımlama
# ==========================================
prompt = ChatPromptTemplate.from_template("""Aşağıdaki bağlamı (context) kullanarak soruyu cevapla:

Bağlam:
{context}

Soru: {input}

Cevap:""")

# ==========================================
# NOT: Buradan sonra bir LLM (Örn: OpenAI, Groq)
# modelini zincire (rag_chain) bağlama
# ==========================================
print(f"Başarılı! {len(splits)} metin parçası hafızadaki vektör veritabanına işlendi.")


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Başarılı! 1470 metin parçası hafızadaki vektör veritabanına işlendi.


7.Hücre - Parçaların okunabilirliğinin kontrolü (ilk 3 parça kontrol)

In [10]:
# Üretilen parçalardan ilk 3 tanesini ekrana yazdırarak kontrol edelim
for i, chunk in enumerate(splits[:3]):
    print(f"=== PARÇA {i+1} ===")
    print(f"📄 Kaynak Sayfa: {chunk.metadata.get('page', 0) + 1}") # Sayfa indeksi 0'dan başlar, +1 ekliyoruz
    print(f"📏 Karakter Uzunluğu: {len(chunk.page_content)}")
    print("📝 İÇERİK:")
    print(chunk.page_content)
    print("=" * 30 + "\n")


=== PARÇA 1 ===
📄 Kaynak Sayfa: 1
📏 Karakter Uzunluğu: 765
📝 İÇERİK:
T.C. 
Resmî Gazete 
Kuruluş Tarihi : ( 7 Teşrinievvel 1336 ) — 7 Ekim 1920 
Yönetim ve yazı işleri için 
Başbakanlık Mevzuatı Geliştirme ve 
Yayın Genel Müdürlüğüne 
başvurulur. 
24 Haziran 1990 
PAZAR 
Sayı : 20558 
YÜRÜTME V E İDARE B O L U M U 
Milletlerarası Sözleşme 
Karar Sayısı ; 90/441 
Ekli "Denizlerin Gemiler Tarafından Kirletilmesinin önlenmesine Ait Uluslararası Söz­
leşme (MARPOL-73 Sözleşmesi)", Sözleşmeyi tadil eden Protokol ( M A R P O L - 7 8 Protokolü) 
ve anılan protokole ilişkin I, II ve V numaralı ekler ile 1984, 1985 ve 1987 tarihlerinde yapılan 
değişikliklerine katılmamız ve 13/9/1989 tarihli ve 89/14547 sayılı Kararnamenin yürürlükten 
kaldırılması; Dışişleri Bakanlığı'nın 30/3/1990, 12/4/1990 tarihli ve 1750, 2022 sayılı yazıları

=== PARÇA 2 ===
📄 Kaynak Sayfa: 1
📏 Karakter Uzunluğu: 789
📝 İÇERİK:
değişikliklerine katılmamız ve 13/9/1989 tarihli ve 89/14547 sayılı Kararnamenin yürürlükten 
k

8.Hücre - Vektör veri tabanı ve Embedding Kontrolü

In [11]:
# 1. InMemory veritabanındaki toplam parça (doküman) sayısını kontrol edelim
toplam_kayit = len(vectorstore.store.keys())
print(f"Veritabanındaki Toplam Vektör Sayısı: {toplam_kayit}\n")

# 2. Örnek bir sorunun nasıl sayısallaştırıldığını (Embedding) görelim
ornek_soru = "Sözleşmenin süresi ne kadar?"
soru_vektoru = embeddings.embed_query(ornek_soru)

print("🔢 Soru Metninin Sayısal (Embedding) Karşılığı:")
print(f"• Vektör Boyutu (Boyut Sayısı): {len(soru_vektoru)}")
print(f"• İlk 5 Sayısal Değer: {soru_vektoru[:5]} ... (Devam ediyor)")
print("-" * 50 + "\n")

# 3. Veritabanının benzerlik aramasını (En Basit Yöntemle) test edelim
print(f"🔍 '{ornek_soru}' Sorusu İçin En Yakın 2 Parçanın Bulunması:")
# En kararlı ve basit arama fonksiyonu
en_yakin_parcalar = vectorstore.similarity_search(ornek_soru, k=2)

for i, doc in enumerate(en_yakin_parcalar):
    print(f"=== EŞLEŞME {i+1} ===")
    print(f"📄 Kaynak Sayfa: {doc.metadata.get('page', 0) + 1}")
    print(f"📝 İlgili Metin İçeriği:\n{doc.page_content}")
    print("=" * 30 + "\n")


Veritabanındaki Toplam Vektör Sayısı: 1470

🔢 Soru Metninin Sayısal (Embedding) Karşılığı:
• Vektör Boyutu (Boyut Sayısı): 512
• İlk 5 Sayısal Değer: [-0.020963139832019806, -0.02955142967402935, -0.03998048976063728, -0.01644560694694519, -0.06474047899246216] ... (Devam ediyor)
--------------------------------------------------

🔍 'Sözleşmenin süresi ne kadar?' Sorusu İçin En Yakın 2 Parçanın Bulunması:
=== EŞLEŞME 1 ===
📄 Kaynak Sayfa: 217
📝 İlgili Metin İçeriği:
which period shall be not less than ten months, unless within that 
period an objection is communicated to the Organization by not 
less than one-third of the Parties or by the Parties the combined 
merchant fleets of which constitute not less than fifty per cent of 
the gross tonnage of the world's merchant fleet whichever condition 
is fulfilled; 
(iv) 
an amendment to Protocol I to the Convention shall be subject to 
the same procedures as for the amendments to the Annexes to the 
Convention, as provided for in sub-parag


**Eşleşme 1 Neden Seçildi? (Güçlü Benzerlik)**

Embedding modeli (yapay zeka) bu metni doğrudan zaman, geçerlilik ve süre kavramlarıyla eşleştirmiştir. Metnin içinde geçen şu ifadeler sorunuzun koordinatlarına çok yakındır:

• "... tarihine kadar ... geçerlidir" cümlesi doğrudan bir süreyi, son kullanma tarihini ve müddeti belirtir.

• "Sözleşmenin Ek 1... hükümlerine uygun olduğunu onaylar" ifadesi aradığınız "sözleşme" kelimesini içerir.

• Alt kısımdaki "Veriliş Tarihi" ve "19.." gibi alanlar zaman haritasında sorunuzla yan yana durur.

Yapay zeka, kelimeler birebir aynı olmasa bile bu metnin "bir şeyin süresinden ve geçerliliğinden" bahsettiğini anlayıp 1. sıraya koymuştur.

**Eşleşme 2 Neden Seçildi? (Zayıf Eşleşme / Hata)**

Vektör veri tabanları siz ne sorarsanız sorun, elindeki en yakın sonuçları getirmek zorundadır. 2. metin aslında bir gemi tablosudur ve süreyle ilgisi yoktur. Ancak sistem bunu şu yüzden 2. sıraya almıştır:

• Metnin başında "Gemide Tamamlanacak" ifadesi yer alıyor. Model, "tamamlanmak" eylemini zamansal bir süreç (bir sürenin bitmesi) olarak yorumlamış olabilir.

• Başka yakın parça bulamadığı için en azından yapısal olarak benzer olan bu dokümanı listeye eklemiştir.

9.Hücre - TEST (10 adet soru & cevap)

In [12]:
import time
from langchain_groq import ChatGroq
from groq import RateLimitError

# ==========================================
# 1. Rastgele Seçilmiş 10 Adet Soru Listesi
# ==========================================
altin_veri_seti = [
    {"soru": "Mahkeme kaç üyeden oluşur?", "dogru_cevap": "Üç üyeden oluşur."},
    {"soru": "Taraflar kaç hakem atar?", "dogru_cevap": "Her taraf bir hakem atar."},
    {"soru": "Üçüncü hakemin atanması kimlerin mutabakatıyla gerçekleşir?", "dogru_cevap": "İki hakemin ortak mutabakatı ile."},
    {"soru": "Gemideki 'slop' tankı nedir ve hangi amaçla kullanılır?", "dogru_cevap": "Slop tankı, geminin içindeki kirli su, yağ ve diğer atık sıvıları toplamak için kullanılan tanktır. Bu sıvılar, geminin çevresel standartlara uygun şekilde bertaraf edilmek üzere depolanır."},
    {"soru": "'Yük tankı' ve 'bulaşık tankı' arasındaki temel fark nedir?", "dogru_cevap": "Yük tankı, geminin taşıdığı yükü (örneğin petrol, kimyasal madde) depolamak için kullanılırken, bulaşık tankı geminin içindeki kirli su ve yağ gibi atık sıvıları toplamak için kullanılır."},
    {"soru": "Bir geminin 'slop' tankı 150 m³ kapasiteye sahipse, bu tankın 80% dolu olması durumunda kaç m³ sıvı bulunur?", "dogru_cevap": "150 m³ × 0.80 = 120 m³."},
    {"soru": "MARPOL 73/78 protokolü kapsamında 'atık su' için hangi tank türü kullanılır?", "dogru_cevap": "“Atık su” için “atık su tankı” (waste water tank) kullanılır. Bu tank, geminin içindeki su atıklarını depolamak için ayrılmıştır."},
    {"soru": "MARPOL 73/78 protokolü, gemi atıklarının hangi iki ana kategoriye ayrılmasını öngörür?", "dogru_cevap": "“Atık su” (waste water) ve “atık yağ” (waste oil) olmak üzere iki ana kategori."},
    {"soru": "Bir geminin 'slop' tankı 300 m³ kapasiteye sahipse, bu tankın 90% dolu olması durumunda kaç m³ sıvı bulunur?", "dogru_cevap": "300 m³ × 0.90 = 270 m³."},
    {"soru": "'Bulaşık tankı'nın derinliği 5 m ise, 250 m³’lük bir hacim elde etmek için genişliği ve yüksekliği ne kadar olmalıdır?", "dogru_cevap": "250 = G × H × 5 → G × H = 50. Örneğin, genişlik 5 m, yüksekliği 10 m (5×10=50) olabilir."}
]

# ==========================================
# 2. Model ve Yardımcı Fonksiyonların Tanımı
# ==========================================
hakem_llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.0)

def invoke_with_retry(llm_instance, prompt_text, retries=5, delay=5):
    """Rate Limit (429) hatası alındığında bekleyip otomatik tekrar deneyen güvenli çağrı fonksiyonu"""
    for i in range(retries):
        try:
            return llm_instance.invoke(prompt_text).content
        except Exception as e:
            if "429" in str(e) or isinstance(e, RateLimitError):
                wait_time = delay * (i + 1)
                print(f"\n⚠️ Rate limit aşıldı! {wait_time} saniye bekleniyor (Deneme {i+1}/{retries})...")
                time.sleep(wait_time)
            else:
                raise e
    raise RuntimeException("Maksimum deneme sınırına ulaşıldı, API kotası çok yoğun.")

def test_bot(soru):
    docs = retriever.invoke(soru)
    context = "\n\n".join(d.page_content for d in docs)
    prompt = f"Bağlam:\n{context}\n\nSoru: {soru}\nCevap:"
    return invoke_with_retry(llm, prompt)

print(f"CANLI TEST BAŞLATILDI. Belirlenen {len(altin_veri_seti)} soru sırayla analiz ediliyor (Hata Korumalı sürüm):\n")

# ==========================================
# 3. 🔄 Test Döngüsü
# ==========================================
for idx, senaryo in enumerate(altin_veri_seti):
    soru = senaryo["soru"]
    gercek = senaryo["dogru_cevap"]

    # Güvenli bot cevabı üretimi
    bot_cevap = test_bot(soru)

    # Hakem İstem Şablonu
    hakem_prompt = f"""Soru: {soru}
Gerçek Doğru Cevap: {gercek}
Öğrenci Cevabı: {bot_cevap}

Yukarıdaki öğrenci cevabına sadece 1 ile 5 arasında bir rakam notu ver ve yanına çok kısa tek cümle gerekçeni yaz.
Örnek format: '4 - Anlamca doğru fakat kelimeler eksik.'"""

    # Güvenli hakem kararı üretimi
    hakem_karari = invoke_with_retry(hakem_llm, hakem_prompt)

    # Sonuçları ekrana yansıtıyoruz
    print(f"📝 [{idx+1}/{len(altin_veri_seti)}] Soru: {soru}")
    print(f"🤖 Botun Yanıtı: {bot_cevap}")
    print(f"👨‍⚖️ Hakem Kararı: {hakem_karari}")
    print("="*60 + "\n")

    time.sleep(3.5) # Groq rate-limit koruma süresi 3.5 saniyeye yükseltildi

print(f"🏁 Belirlenen {len(altin_veri_seti)} soruluk test başarıyla tamamlandı!")

CANLI TEST BAŞLATILDI. Belirlenen 10 soru sırayla analiz ediliyor (Hata Korumalı sürüm):

📝 [1/10] Soru: Mahkeme kaç üyeden oluşur?
🤖 Botun Yanıtı: Mahkeme üç üyeden oluşur.
👨‍⚖️ Hakem Kararı: 5 - Tam doğru ve eksiksiz.

📝 [2/10] Soru: Taraflar kaç hakem atar?
🤖 Botun Yanıtı: Taraflar her biri bir hakem atar; yani toplamda iki hakem atılır.
👨‍⚖️ Hakem Kararı: 5 - Tam olarak doğru, her taraf bir hakem atar.

📝 [3/10] Soru: Üçüncü hakemin atanması kimlerin mutabakatıyla gerçekleşir?
🤖 Botun Yanıtı: Üçüncü hakemin atanması, **her iki tarafın atadığı ilk iki hakemin ortak mutabakatıyla** gerçekleşir.
👨‍⚖️ Hakem Kararı: 5 - Tam olarak doğru, iki hakemin ortak mutabakatıyla gerçekleşir.

📝 [4/10] Soru: Gemideki 'slop' tankı nedir ve hangi amaçla kullanılır?
🤖 Botun Yanıtı: **Slop tankı nedir?**  
Gemideki “slop tankı”, geminin yapısına gömülü, özellikle tank drenajları, tank yıkama sıvıları ve diğer yağlı karışımların toplanması için özel olarak tasarlanmış bir tanktır.  

**Kullanım amacı**

***TEST SONUCU:***
(test sonucları her çalışmada değişkenlik gösteriyor)

*Toplam Soru: 10 adet*

* Doğru: 7 adet
* Yanlış: 1 adet
* Boş: 1 adet
* Esik/Kısmen doğru: 1 adet
----------------------------
* 5 puan: 7 adet
* 4 puan: YOK
* 3 puan: YOK
* 2 puan: 1 adet
* 1 puan: 2 adet

**!!! Test sonucuna göre , prompt güncelleme ve chunk değer değişiklikleri ? yapalım !!!**

10.Hücre - STREAMLIT arayüz hazırlık ve Tünelleme

In [13]:
%%writefile app.py
import os, streamlit as st
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

st.set_page_config(layout="wide")

# 🎨 Deniz Temizliği ve Denizcilik Teması Renk Ayarları (CSS)
st.markdown("""
    <style>
    .stApp { background-color: #f0f5f9; }
    h1 { color: #0a4b7c !important; }
    .stChatMessage { border-radius: 12px; padding: 10px; margin-bottom: 8px; }
    div[data-testid="stChatMessageUser"] { background-color: #e0f2fe !important; border-left: 5px solid #0284c7; }
    div[data-testid="stChatMessageAssistant"] { background-color: #ffffff !important; border-left: 5px solid #0f172a; border: 1px solid #e2e8f0; }
    </style>
""", unsafe_allow_html=True)

st.title("🌊 MARPOL RAG Chatbot")

@st.cache_resource
def init_fast_rag(uploaded_file):
    temp_path = "temp.pdf"
    with open(temp_path, "wb") as f:
        f.write(uploaded_file.getbuffer())

    # Hızlı PDF Okuyucu (PyMuPDFLoader)
    loader = PyMuPDFLoader(temp_path)
    splits = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=250
    ).split_documents(loader.load())

    # 🚀 ÇOK DAHA HIZLI İNEN VE TÜRKÇE DESTEKLEYEN MODEL
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/distiluse-base-multilingual-cased-v1")
    vectorstore = InMemoryVectorStore.from_documents(splits, embeddings)

    # Arama kapsamı
    retriever = vectorstore.as_retriever(search_kwargs={"k": 8})

    # LLM MODEL
    llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.0)

    # 🛡️ GÜNCELLEME: Hem dış bilgi kısıtlaması hem de OCR düzeltme kuralları entegre edildi.
    prompt = ChatPromptTemplate.from_template("""Sen sadece sana verilen bağlam (context) belgelerine dayanarak yanıt veren profesyonel bir denizcilik asistanısın.

Kritik Kurallar:
1. SADECE sana sunulan 'Bağlam' içindeki bilgilere dayanarak soruyu cevapla.
2. Eğer sorulan sorunun cevabı verilen bağlam içerisinde kesinlikle YOKSA veya bağlamdan mantıklı bir çıkarım yapılamıyorsa, kesinlikle kendi genel bilgini kullanma! Doğrudan şu yanıtı ver: "Bu sorunun cevabı yüklenen belgede bulunmamaktadır."
3. Bağlam dışından asla varsayımda bulunma, tahminde bulunma veya dışarıdan ek bilgi uydurma.
4. Bağlam taranmış resmi belgelerden OCR ile aktarıldığı için bazı kelimeler birleşmiş (örn: 'MilletlerarasıSözleşme', 'bulaşıktankı') veya harfler hatalı okunmuş olabilir (örn: 'önlenmesine' yerine 'onlenmesine'). Bu tür yazım ve birleşme hatalarını zihninde düzelterek anlamlandır, ancak belgede geçmeyen hiçbir sayıyı veya veriyi asla değiştirme.
5. Yanıtlarını sadece Türkçe olarak ver.

Bağlam:
{context}

Soru: {input}
Cevap:""")

    chain = (
        {"context": retriever | (lambda docs: "\n\n".join(d.page_content for d in docs)), "input": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    return chain, len(splits)

uploaded_file = st.file_uploader("⚓ Analiz edilecek denizcilik belgesini buraya bırakın", type=["pdf"])

if uploaded_file:
    rag_chain, split_count = init_fast_rag(uploaded_file)
    st.success(f"💧 Belge saniyeler içinde hafızaya alındı ({split_count} parça)! Yapay zeka hazır.")

    if "messages" not in st.session_state: st.session_state.messages = []
    for msg in st.session_state.messages: st.chat_message(msg["role"]).write(msg["content"])

    if user_query := st.chat_input("Sorunuzu buraya yazın..."):
        st.chat_message("user").write(user_query)
        st.session_state.messages.append({"role": "user", "content": user_query})

        with st.chat_message("assistant"):
            with st.spinner("Metin derinliklerinde aranıyor..."):
                response = rag_chain.invoke(user_query)
                st.write(response)
        st.session_state.messages.append({"role": "assistant", "content": response})

Overwriting app.py


11.Hücre - Sunucu çalıştır

In [18]:
import threading
from google.colab import output

# 1. Port çakışmalarını tamamen temizlemek için uygulamayı 8540 portunda başlat
def run_streamlit():
    !streamlit run app.py \
        --server.port 8540 \
        --server.enableCORS=false \
        --server.enableXsrfProtection=false \
        --server.maxUploadSize=500 \
        --server.fileWatcherType=none

threading.Thread(target=run_streamlit).start()
print("⚡ Akış kanalı optimize edildi. Uygulama hücre altına yükleniyor...")

# 2. Google'ın Axios ağ hatalarını engelleyen resmi iç köprüsünü çalıştırıyoruz
# height=800 parametresi dosyaları rahat yüklemeniz için ekranı daha da büyütür.
output.serve_kernel_port_as_iframe(8540, height=800)


⚡ Akış kanalı optimize edildi. Uygulama hücre altına yükleniyor...


<IPython.core.display.Javascript object>

### 🌐 12. Hücre - Masaüstü Uygulaması İçin Tünel Adresi Oluşturma

Masaüstü uygulamanızda kullanacağınız tünel adresini almak için aşağıdaki hücreyi çalıştırmanız yeterlidir. Hücreyi çalıştırdığınızda size özel şifresiz, temiz bir web adresi üretilecektir.

In [ ]:
# 1. Ücretsiz tünelleme aracı olan localtunnel paketini kuruyoruz
!npm install -g localtunnel -q

# 2. IP adresimizi öğreniyoruz (Giriş yaparken güvenlik şifresi olarak isteyecektir)
import urllib
print("🔑 Giriş Şifreniz (IP Adresiniz):", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip())
print("--------------------------------------------------")

# 3. Streamlit uygulamamızı (8540 portu) dış dünyaya güvenli tünel ile açıyoruz
!lt --port 8540

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼
changed 22 packages in 1s
⠼
⠼3 packages are looking for funding
⠼  run `npm fund` for details
⠼🔑 Giriş Şifreniz (IP Adresiniz): 34.106.249.79
--------------------------------------------------
your url is: https://lovely-poems-exist.loca.lt


14.Hücre - COLAB ile arayüz oluşturma (streamlit alternatif )

In [20]:
import ipywidgets as widgets
from IPython.display import display, HTML
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 🎨 1. Deniz Temizliği Teması (CSS)
display(HTML("""
<style>
    .chat-box { background-color: #f0f5f9; border-radius: 12px; padding: 15px; max-height: 400px; overflow-y: auto; border: 1px solid #bae6fd; }
    .msg-user { background-color: #e0f2fe; border-left: 5px solid #0284c7; padding: 8px 12px; margin: 5px 0; border-radius: 8px; font-family: sans-serif; color: #0369a1; }
    .msg-bot { background-color: #ffffff; border-left: 5px solid #0f172a; padding: 8px 12px; margin: 5px 0; border-radius: 8px; font-family: sans-serif; border: 1px solid #e2e8f0; color: #1e293b; }
    .title-banner { color: #0a4b7c; font-family: sans-serif; font-weight: bold; font-size: 20px; margin-bottom: 10px; }
</style>
"""))

# 📺 2. Arayüz Elementleri
baslik = widgets.HTML('<div class="title-banner">🌊 MARPOL RAG Chatbot (PDF Kilitli)</div>')
sohbet_ekrani = widgets.HTML(value='<div class="chat-box" id="chat-box">🤖 <i>Lütfen sözleşme hakkında spesifik bir kural veya madde sorun...</i></div>')
girdi_kutusu = widgets.Text(placeholder='Sorunuzu buraya yazın ve Enter\'a basın...', layout=widgets.Layout(width='80%'))
gonder_butonu = widgets.Button(description='⚓ Sor', button_style='info', layout=widgets.Layout(width='18%'))

mesaj_gecmisi = []

# ⚡ 3. OCR Toleranslı ve Sözleşmeye Kilitli RAG Motoru
def soru_cevapla(sender):
    soru = girdi_kutusu.value.strip()
    if not soru:
        return

    girdi_kutusu.value = "Okyanus derinliklerinde aranıyor..."

    try:
        # Arama motorunu genişleterek k=6 parça çekiyoruz (OCR hatalarını kaçırmamak için)
        retriever.search_kwargs["k"] = 6
        docs = retriever.invoke(soru)
        context = "\n\n".join(d.page_content for d in docs)

        # 🧠 Botun serbest konuşmasını engelleyen uzman prompt şablonu
        uzman_prompt = ChatPromptTemplate.from_template("""Aşağıdaki bağlamı (context) bir denizcilik uzmanı gibi incele ve soruyu sadece bu bağlama bağlı kalarak Türkçe cevapla.
Sana verilen bu bağlam resmi sözleşme belgelerinden OCR ile aktarıldığı için imla hataları içerebilir; kelimelerin gelişinden anlamı zihninde düzelt.

Kritik Kural: Bağlam dışına asla çıkma. Eğer aranan bilgi sana sunulan aşağıdaki metinlerin içinde kesin olarak geçmiyorsa, serbestçe uydurma; dürüstçe "Bu bilgi yüklenen sözleşme içeriğinde yer almamaktadır" de.

Bağlam:
{context}

Soru: {input}
Cevap:""")

        # Kesin kilitli zincir tetikleniyor
        zincir = uzman_prompt | llm | StrOutputParser()
        cevap = zincir.invoke({"context": context, "input": soru})

    except Exception as e:
        cevap = f"⚠️ Yapay zeka motoruna bağlanırken bir hata oluştu: {str(e)}"

    mesaj_gecmisi.append(f'<div class="msg-user"><b>⚓ Siz:</b> {soru}</div>')
    mesaj_gecmisi.append(f'<div class="msg-bot"><b>🤖 Bot:</b> {cevap}</div>')

    sohbet_ekrani.value = f'<div class="chat-box">{"".join(mesaj_gecmisi)}</div>'
    girdi_kutusu.value = ""

girdi_kutusu.on_submit(soru_cevapla)
gonder_butonu.on_click(soru_cevapla)

arayuz_kutusu = widgets.VBox([baslik, sohbet_ekrani, widgets.HBox([girdi_kutusu, gonder_butonu])])
display(arayuz_kutusu)





*** 9.hücredeki test döngüsü ile 14.hücredeki Colab arayüzü (UI) arasında aynı soruda farklı sonuçların temel nedenleri:***
1. Aşırı Katı Prompt (Talimat) Kısıtlaması (En Büyük Neden)
2. Geri Getirilen Parça Sayısı (k Parametresi)



In [ ]:
# Gerekli kütüphane yüklü değilse arka planda hızlıca kuralım
!pip install -q gradio

import gradio as gr
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 🌊 RAG Sorgu ve Cevaplama Fonksiyonu
def rag_sohbet_robotu(mesaj, gecmis):
    if not mesaj.strip():
        return ""

    try:
        # OCR toleransı ve zengin içerik için en alakalı 6 parçayı çekiyoruz
        retriever.search_kwargs["k"] = 6
        docs = retriever.invoke(mesaj)
        context = "\n\n".join(d.page_content for d in docs)

        # Sıkı kurallara bağlı uzman prompt şablonumuz
        uzman_prompt = ChatPromptTemplate.from_template("""Aşağıdaki bağlamı (context) bir denizcilik uzmanı gibi incele ve soruyu sadece bu bağlama bağlı kalarak Türkçe cevapla.
Sana verilen bu bağlam resmi sözleşme belgelerinden OCR ile aktarıldığı için imla hataları içerebilir; kelimelerin gelişinden anlamı zihninde düzelt.

Kritik Kural: Bağlam dışına asla çıkma. Eğer aranan bilgi sana sunulan aşağıdaki metinlerin içinde kesin olarak geçmiyorsa, serbestçe uydurma; dürüstçe "Bu bilgi yüklenen sözleşme içeriğinde yer almamaktadır" de.

Bağlam:
{context}

Soru: {input}
Cevap:""")

        zincir = uzman_prompt | llm | StrOutputParser()
        cevap = zincir.invoke({"context": context, "input": mesaj})
        return cevap
    except Exception as e:
        return f"⚠️ Yapay zeka motoruna bağlanırken bir hata oluştu: {str(e)}"

# 🎨 Gradio 6.0 Uyumlu Modern Arayüz Tasarımı
with gr.Blocks() as arayuz:
    gr.Markdown("""
    # <center>🌊 DENİZCİLİK RAG Chatbot</center>
    ### <center>
    Yapay Zeka Asistanı</center>
    """)

    # Gradio ChatInterface için en sade ve sorunsuz parametre kombinasyonu
    chatbot = gr.ChatInterface(
        fn=rag_sohbet_robotu,
        chatbot=gr.Chatbot(height=500, placeholder="🤖 <i>Lütfen sözleşme
        autofocus=Truehakkında spesifik bir kural, madde veya formül sorun...</i>"),
        textbox=gr.Textbox(placeholder="Sorunuzu buraya yazın ve Enter'a basın...", container=False, scale=7),
    )

# 🚀 Uygulamayı başlatıyoruz ve Gradio 6 temamızı launch kısmına devrediyoruz
arayuz.launch(theme=gr.themes.Soft(primary_hue="blue", secondary_hue="slate"), demo.launch(), debug=False)